# DAX From Scratch: A Complete Guide

## Table of Contents
1. What is DAX and Why It Exists
2. The Data Model (you can't learn DAX without this)
3. DAX Syntax Basics
4. Three Ways to Write DAX: Columns, Measures, Tables
5. The Most Important Concept: Row Context vs. Filter Context
6. CALCULATE - The Heart of DAX
7. Core Function Families
8. Iterator Functions (the X functions)
9. Time Intelligence
10. Variables (VAR/RETURN)
11. Common Mistakes & Best Practices
12. Practice Exercises

---

## 1. What is DAX and Why It Exists

**DAX (Data Analysis Expressions)** is a formula language used in:
- Power BI
- Power Pivot (Excel)
- SQL Server Analysis Services (Tabular mode)

It looks a bit like Excel formulas, which helps beginners get started but also causes confusion - **DAX is not Excel**. Excel formulas operate on individual cells. DAX formulas operate on **entire columns and tables**, and their result depends on **context** (more on that soon - it's the single biggest idea in DAX).

DAX exists to let you create:
- **Calculated columns** - new columns added row-by-row to a table
- **Measures** - dynamic calculations that respond to whatever filters are applied (used in visuals, PivotTables, cards)
- **Calculated tables** - entire new tables generated by a formula

Think of DAX as the "brain" that sits on top of your data model and answers questions like: *"What's total sales for this product category, in this region, in this quarter, compared to last year?"*

---

## 2. The Data Model (You Can't Learn DAX Without This)

DAX doesn't work on a single flat spreadsheet the way Excel does. It works on a **relational data model** - multiple tables connected by relationships. This is critical because DAX's behavior depends entirely on how tables relate to each other.

### Typical structure: Star Schema
- **Fact table** - the big table with transactional/numeric data (e.g., `Sales`: Date, ProductID, CustomerID, Quantity, Amount)
- **Dimension tables** - smaller descriptive tables (e.g., `Product`, `Customer`, `Date`, `Region`)

Relationships connect a dimension's primary key to the fact table's foreign key:

```
Product[ProductID] (1) ----< (many) Sales[ProductID]
Customer[CustomerID] (1) ----< (many) Sales[CustomerID]
Date[Date] (1) ----< (many) Sales[Date]
```

**Why this matters for DAX:** relationships determine how filters "flow." If you filter `Product[Category] = "Bikes"`, that filter flows across the relationship into `Sales`, restricting which sales rows are visible to your calculation. This flow of filters *is* filter context, which we'll cover in Section 5.

**Rule of thumb:** always build a proper Date table and mark it as a Date table. Almost all time intelligence functions require this.

---

## 3. DAX Syntax Basics

### Referencing columns and tables
```dax
TableName[ColumnName]
```
Always fully qualify columns with their table name. It's not strictly required everywhere, but it's best practice and required in many contexts (like inside CALCULATE).

### Referencing measures
```dax
[MeasureName]
```
Measures are *not* qualified with a table name when referenced (even though they're "stored" under a table in the model).

### Basic syntax example
```dax
Total Sales = SUM(Sales[Amount])
```
This defines a measure called `Total Sales` equal to the sum of the `Amount` column in the `Sales` table.

### Data types in DAX
- Whole Number / Decimal Number
- Currency (fixed decimal, avoids floating point rounding issues)
- Date/DateTime
- Text (String)
- Boolean (TRUE/FALSE)
- Blank (DAX's version of NULL - written as `BLANK()`)

### Operators
| Type | Operators |
|---|---|
| Arithmetic | `+  -  *  /  ^` |
| Comparison | `=  <>  >  <  >=  <=` |
| Text concatenation | `&` |
| Logical | `&&` (AND), `\|\|` (OR), `NOT()` |

Example:
```dax
Full Name = Customer[FirstName] & " " & Customer[LastName]
```

### Comments
```dax
// single line comment
/* multi
   line comment */
```

---

## 4. Three Ways to Write DAX

### A. Calculated Columns
Computed **once per row**, stored physically in the table, refreshed only when the model refreshes.

```dax
Profit = Sales[Amount] - Sales[Cost]
```

- Evaluated row-by-row (this uses **row context** - see Section 5)
- Takes up memory (it's materialized data)
- Use when you need the value to act like a normal column (e.g., to slice/filter/group by it)

### B. Measures
Computed **on the fly**, whenever used in a visual, and their result changes depending on filters applied (year selected, region clicked, etc.).

```dax
Total Profit = SUM(Sales[Amount]) - SUM(Sales[Cost])
```

- Evaluated using **filter context** (Section 5)
- Not stored - calculated at query time
- This is where 90% of real DAX work happens
- Best practice: organize measures in a dedicated "Measures table" (a small disconnected table) for tidiness

### C. Calculated Tables
An entire new table generated by a DAX expression, useful for things like a manually built Date table, or a summarized/aggregated table.

```dax
Date = CALENDAR(DATE(2020,1,1), DATE(2026,12,31))
```

---

## 5. The Most Important Concept: Row Context vs. Filter Context

This is the concept that separates people who "sort of" know DAX from people who actually understand it. Nearly every confusing DAX bug traces back to a misunderstanding here.

### Row Context
Row context means: *"DAX currently knows which single row it's looking at."*

This exists automatically in **calculated columns**, because a calculated column is evaluated once per row.

```dax
Sales[Profit] = Sales[Amount] - Sales[Cost]
```
For every row, DAX looks at *that row's* Amount and Cost. It has "walked into" the row.

Row context also appears inside **iterator functions** like `SUMX` (see Section 8), even when used in a measure.

### Filter Context
Filter context means: *"Given everything currently filtering the data - slicers, visual rows/columns, other filters - what subset of rows is visible right now?"*

This is what makes measures dynamic. The exact same measure:
```dax
Total Sales = SUM(Sales[Amount])
```
returns a different number depending on filter context:
- On a card with no filters → total of *all* sales
- In a table broken out by Year and Region → total *for that year and region combination*
- With a slicer set to "Bikes" → total *only for bikes*

The filter context is invisible in the formula itself - it comes from *where* the measure is used (which row/column of a visual, which slicer selections are active).

### Why this distinction matters
A calculated column has NO filter context awareness at creation time - it's baked in at refresh, row by row, ignoring anything the user later filters on in a report.
A measure has NO fixed value - it doesn't "mean" anything until it's placed in a context (a cell in a visual, a card, etc.).

### Context Transition (the tricky bridge)
Sometimes row context needs to become filter context. This happens automatically whenever you use `CALCULATE` (or an iterator like `SUMX`) inside a row context - DAX takes "the current row" and turns it into "a filter restricting to that row's values." This is called **context transition** and is one of the more advanced-but-critical ideas once you move past basics.

```dax
Sales[RunningMarginPct] =
DIVIDE(
    Sales[Amount] - Sales[Cost],
    CALCULATE(SUM(Sales[Amount]))   -- context transition happens here
)
```

---

## 6. CALCULATE - The Heart of DAX

If DAX has one signature function, it's `CALCULATE`. It's the only function that can **change filter context**. Everything involving "compare to last year," "% of total," "ignore this filter," etc. runs through CALCULATE.

### Syntax
```dax
CALCULATE(<expression>, <filter1>, <filter2>, ...)
```

- `<expression>` - usually an aggregation like SUM, AVERAGE, or another measure
- `<filter>` arguments - conditions that modify the filter context before the expression is evaluated

### Example: Filter Override
```dax
Bike Sales = CALCULATE([Total Sales], Product[Category] = "Bikes")
```
This ignores whatever category is currently in context and forces it to "Bikes."

### Example: Removing filters with ALL
```dax
Total Sales All Products =
CALCULATE([Total Sales], ALL(Product))
```
`ALL(Product)` strips away any filters coming from the Product table, so you get the grand total regardless of what's sliced.

This pattern - `[Measure] / CALCULATE([Measure], ALL(Table))` - is exactly how you build **"% of total"** calculations.

```dax
Pct of Total Sales =
DIVIDE([Total Sales], CALCULATE([Total Sales], ALL(Product)))
```

### Common filter-modifier functions used inside CALCULATE
| Function | Purpose |
|---|---|
| `ALL(Table or Column)` | Removes filters entirely |
| `ALLEXCEPT(Table, Column)` | Removes all filters on a table except the ones listed |
| `ALLSELECTED()` | Removes filters added by the visual, but keeps external slicers |
| `FILTER(Table, condition)` | Builds a custom filtered table to apply |
| `KEEPFILTERS()` | Adds a filter without replacing existing ones on that column |

---

## 7. Core Function Families

### Aggregation
`SUM`, `AVERAGE`, `MIN`, `MAX`, `COUNT`, `COUNTROWS`, `DISTINCTCOUNT`

### Logical
`IF(condition, true_result, false_result)`, `SWITCH(expression, value1, result1, value2, result2, ..., else)`, `AND`, `OR`, `NOT`

```dax
Sales Tier =
SWITCH(
    TRUE(),
    [Total Sales] > 100000, "Gold",
    [Total Sales] > 50000, "Silver",
    "Bronze"
)
```

### Text
`CONCATENATE`, `LEFT`, `RIGHT`, `MID`, `LEN`, `UPPER`, `LOWER`, `TRIM`, `FORMAT`

### Date/Time
`TODAY()`, `NOW()`, `YEAR()`, `MONTH()`, `DAY()`, `DATEDIFF()`, `EOMONTH()`

### Relationship functions
`RELATED(Column)` - pulls a value across a relationship into a row context (used in calculated columns)
`RELATEDTABLE(Table)` - the reverse: pulls related rows from the "many" side

```dax
Product[CategoryName] = RELATED(Category[CategoryName])
```

---

## 8. Iterator Functions (the "X" Functions)

These are functions that loop **row by row** over a table, even when used inside a measure. This is where row context can appear inside a measure.

| Function | What it does |
|---|---|
| `SUMX(Table, expression)` | Evaluates expression per row, then sums results |
| `AVERAGEX` | Same, but averages |
| `MAXX` / `MINX` | Same, but max/min |
| `COUNTX` | Counts rows where expression is non-blank |
| `RANKX` | Ranks a value against a table of values |

### Why iterators matter
Some calculations can't be done with plain SUM because they require row-level multiplication first.

```dax
Total Revenue =
SUMX(Sales, Sales[Quantity] * Sales[UnitPrice])
```

You **cannot** do this with `SUM(Sales[Quantity]) * SUM(Sales[UnitPrice])` - that gives the wrong number, because it multiplies the *totals*, not each row before summing. This is one of the most common beginner mistakes.

---

## 9. Time Intelligence

Time intelligence functions require a proper marked Date table with a continuous, unbroken date range.

| Function | Purpose |
|---|---|
| `TOTALYTD(expr, dates)` | Year-to-date total |
| `SAMEPERIODLASTYEAR(dates)` | Shifts the date filter back one year |
| `DATEADD(dates, n, interval)` | Shifts dates forward/back by n periods |
| `DATESYTD`, `DATESMTD`, `DATESQTD` | Returns YTD/MTD/QTD date sets |
| `PARALLELPERIOD` | Similar shifting, at period granularity |

### Example: Year-over-Year growth
```dax
Sales PY = CALCULATE([Total Sales], SAMEPERIODLASTYEAR('Date'[Date]))

Sales YoY % =
DIVIDE([Total Sales] - [Sales PY], [Sales PY])
```

---

## 10. Variables (VAR/RETURN)

Variables make DAX more readable and more efficient - a variable is computed once and can be reused, instead of recalculating the same expression multiple times.

```dax
YoY Growth =
VAR CurrentSales = [Total Sales]
VAR PriorSales = CALCULATE([Total Sales], SAMEPERIODLASTYEAR('Date'[Date]))
VAR Growth = DIVIDE(CurrentSales - PriorSales, PriorSales)
RETURN
    Growth
```

Benefits:
- Improves readability (self-documenting names)
- Improves performance (avoids recalculating the same sub-expression)
- Makes debugging easier (you can temporarily `RETURN` a variable to inspect it)

---

## 11. Common Mistakes & Best Practices

1. **Not building a real Date table.** Time intelligence functions silently misbehave without one.
2. **Confusing calculated columns with measures.** If the value should change based on report filters/slicers, it should be a measure, not a column.
3. **Multiplying totals instead of using SUMX.** (`SUM(A) * SUM(B)` ≠ `SUMX(Table, A*B)`)
4. **Forgetting DIVIDE().** Use `DIVIDE(numerator, denominator, alternate_result)` instead of `/` - it gracefully handles divide-by-zero instead of erroring.
5. **Overusing calculated columns.** They bloat model size since they're stored. Prefer measures when possible.
6. **Not understanding ALL() vs ALLSELECTED().** `ALL` wipes everything; `ALLSELECTED` respects what the user explicitly chose (like slicers) but ignores the visual's own row/column groupings - the difference matters for subtotal/percent-of-total calcs.
7. **Ignoring relationship direction/cardinality.** Filters only flow the direction relationships allow (by default, one-to-many, single direction from the "1" side to the "many" side).

---

## 12. Practice Exercises

Try these against a simple `Sales`, `Product`, `Date`, `Customer` model:

1. Write a measure `Total Sales` = sum of `Sales[Amount]`.
2. Write a measure `Total Transactions` using `COUNTROWS(Sales)`.
3. Write `Average Order Value` = `DIVIDE([Total Sales], [Total Transactions])`.
4. Write `Sales - Prior Year` using `SAMEPERIODLASTYEAR`.
5. Write `% of Category Total` using `CALCULATE` + `ALLEXCEPT`.
6. Write a calculated column `Order Size Tier` using `SWITCH(TRUE(), ...)`.
7. Rewrite exercise 3 using `VAR/RETURN` for clarity.

Once you're comfortable with these, you've covered the concepts that make up roughly 80% of real-world DAX work. Everything beyond this (advanced iterator patterns, virtual relationships with `TREATAS`, disconnected "what-if" tables, complex context transition chains) builds directly on the row context / filter context foundation from Section 5 - so it's worth going back and re-reading that section once the practice exercises start to click.